<a href="https://colab.research.google.com/github/talhanoor23/Deep-Learning-Experiments/blob/main/X_rays_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
 99% 2.26G/2.29G [00:23<00:00, 99.6MB/s]
100% 2.29G/2.29G [00:23<00:00, 106MB/s] 


In [ ]:
!unzip chest-xray-pneumonia.zip

In [ ]:
# import tensorflow as tf
# from tensorflow.keras.preprocessing.image import ImageDataGenerator

# IMG_SIZE = (224, 224)

# train_datagen = ImageDataGenerator(rescale=1./255,
#                                    rotation_range=10,
#                                    zoom_range=0.1,
#                                    horizontal_flip=True)

# test_datagen = ImageDataGenerator(rescale=1./255)

# train_data = train_datagen.flow_from_directory(
#     "chest_xray/train",
#     target_size=IMG_SIZE,
#     batch_size=32,
#     class_mode='binary'
# )

# val_data = test_datagen.flow_from_directory(
#     "chest_xray/val",
#     target_size=IMG_SIZE,
#     batch_size=32,
#     class_mode='binary'
# )

# test_data = test_datagen.flow_from_directory(
#     "chest_xray/test",
#     target_size=IMG_SIZE,
#     batch_size=32,
#     class_mode='binary'
# )


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Augmentations for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.2
)

# 2. No augmentation for validation
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)


train_data = train_datagen.flow_from_directory(
    '/content/chest_xray/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='training',   # ← uses train_datagen with augmentation
    shuffle=True
)

val_data = val_datagen.flow_from_directory(
    '/content/chest_xray/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    subset='validation',  # ← uses val_datagen without augmentation
    shuffle=False
)


Found 4173 images belonging to 2 classes.
Found 1043 images belonging to 2 classes.


In [ ]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Freeze the base model initially
for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
history = model.fit(train_data,
                    validation_data=val_data,
                    epochs=9)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/9
131/131 ━━━━━━━━━━━━━━━━━━━━ 155s 976ms/step - accuracy: 0.7067 - loss: 0.6146 - val_accuracy: 0.9214 - val_loss: 0.2764
Epoch 2/9
131/131 ━━━━━━━━━━━━━━━━━━━━ 89s 674ms/step - accuracy: 0.8536 - loss: 0.3263 - val_accuracy: 0.9243 - val_loss: 0.2262
Epoch 3/9
131/131 ━━━━━━━━━━━━━━━━━━━━ 89s 676ms/step - accuracy: 0.8794 - loss: 0.2703 - val_accuracy: 0.9434 - val_loss: 0.1952
Epoch 4/9
131/131 ━━━━━━━━━━━━━━━━━━━━ 89s 681ms/step - accuracy: 0.8912 - loss: 0.2660 - val_accuracy: 0.9233 - val_loss: 0.2147
Epoch 5/9
131/131 ━━━━━━━━━━━━━━━━━━━━ 89s 681ms/step - accuracy: 0.9033 - loss: 0.2386 - val_accuracy: 0.9329 - val_loss: 0.1963
Epoch 6/9
131/131 ━━━━━━━━━━━━━━━━━━━━ 142s 683ms/step - accuracy: 0.9087 - loss: 0.2347 - val_accuracy: 0.9367 - val_loss: 0.1844
Epoch 7/9
131/131 ━━━━━━━━━━━━━━━━━━━━ 141s 677ms/step - accuracy: 0.9110 - loss: 0.2255 - val_accuracy: 0.9434 - val_loss: 0.1778
Epoch 8/9
131/131 ━━━━━━━━━━━━━━━━━━━━ 91s 695ms/step - accuracy: 0.9063 - loss: 0.2208

In [ ]:
import os

# Create the directory if it doesn't exist
os.makedirs("saved_model", exist_ok=True)

# Now you can save the model
model.save("saved_model/densenet_finetuned.keras")

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    '/content/chest_xray/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',   # or 'categorical'
    shuffle=False
)

Found 624 images belonging to 2 classes.


In [ ]:
model.evaluate(test_generator)

20/20 ━━━━━━━━━━━━━━━━━━━━ 16s 815ms/step - accuracy: 0.7597 - loss: 0.4768


[0.31281596422195435, 0.8605769276618958]

In [ ]:
# Unfreeze some top layers of the base model
import tensorflow as tf
for layer in base_model.layers[-50:]:
    layer.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),  # smaller LR
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(train_data,
          validation_data=val_data,
          epochs=5)


In [ ]:
model.summary()

In [ ]:
model.evaluate(test_generator)

In [ ]:
import os

# Create the directory if it doesn't exist
os.makedirs("saved_model", exist_ok=True)

# Now you can save the model
model.save("saved_model/more_filtered_densenet_finetuned.keras")